## Kafka Setup on EC2 (Amazon Linux 2) — Guide

### 1. Launch EC2 Correctly

Instance Type (IMPORTANT)

Use minimum:

t3.medium  (4GB RAM)

Do NOT use:

t2.micro

t2.small

Because Kafka default heap = 1GB and small instances crash.

Security Group Rules

Add inbound:

| Type       | Port | Source                    |
| ---------- | ---- | ------------------------- |
| SSH        | 22   | Your IP                   |
| Custom TCP | 9092 | Anywhere (for testing)    |
| Custom TCP | 9093 | Anywhere (for controller) |


2. Connect to EC2

In [ ]:
ssh -i your-key.pem ec2-user@YOUR_PUBLIC_IP

🔷 3. Install Java (Required)

In [ ]:
sudo yum update -y
sudo yum install java-11-amazon-corretto -y

4. Download Kafka (Correct Way)

In [ ]:
curl https://downloads.apache.org/kafka/

In [ ]:
wget https://downloads.apache.org/kafka/3.8.0/kafka_2.13-3.8.0.tgz

In [ ]:
tar -xvf kafka_2.13-3.8.0.tgz
cd kafka_2.13-3.8.0

🔷 5. Fix Memory (Prevent Heap Crash)

In [ ]:
nano bin/kafka-server-start.sh

In [ ]:
export KAFKA_HEAP_OPTS="-Xmx1G -Xms1G"

In [ ]:
export KAFKA_HEAP_OPTS="-Xmx512M -Xms512M"

This prevents: \
Not enough space \
Failed to map 1073741824 bytes

6. Configure KRaft Properly (Most Important Step)

In [ ]:

nano config/kraft/server.properties

Replace important section with:

In [ ]:
node.id=1
process.roles=broker,controller

listeners=PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
advertised.listeners=PLAINTEXT://YOUR_PUBLIC_IP:9092

inter.broker.listener.name=PLAINTEXT
listener.security.protocol.map=PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
controller.listener.names=CONTROLLER

controller.quorum.voters=1@YOUR_PRIVATE_IP:9093

log.dirs=/tmp/kraft-combined-logs

⚠️ VERY IMPORTANT

Get private IP:

hostname -I

Use that for:

controller.quorum.voters

Do NOT use public IP here.

That mistake causes:


Timed out waiting for node assignment
Broker unable to register

7. Clean Storage Before Formatting

In [ ]:
rm -rf /tmp/kraft-combined-logs

8. Generate Cluster ID

In [ ]:
bin/kafka-storage.sh random-uuid

9. Format Storage (Correct Syntax)

In [ ]:
bin/kafka-storage.sh format \
--config config/kraft/server.properties \
--cluster-id YOUR_UUID

10. Start Kafka

In [ ]:
bin/kafka-server-start.sh config/kraft/server.properties

11. Wait 15 Seconds (Controller Election)

12. Create Topic (New SSH Session)

In [ ]:
cd kafka_2.13-3.8.0

bin/kafka-topics.sh --create \
--topic test-topic \
--bootstrap-server localhost:9092 \
--partitions 1 \
--replication-factor 1

Created topic test-topic

13. Test Producer

In [ ]:
bin/kafka-console-producer.sh \
--topic test-topic \
--bootstrap-server localhost:9092

14. Test Consumer

In [ ]:
bin/kafka-console-consumer.sh \
--topic test-topic \
--from-beginning \
--bootstrap-server localhost:9092

✅ Common Errors & Their Fix

| Error                       | Cause                              | Fix                             |
| --------------------------- | ---------------------------------- | ------------------------------- |
| 404 Not Found               | Wrong download URL                 | Use full version path           |
| Not enough space            | Low RAM                            | Reduce heap or upgrade instance |
| Broker shutdown immediately | Wrong controller.quorum.voters     | Use PRIVATE IP                  |
| Topic creation timeout      | Missing inter.broker.listener.name | Add config line                 |
| Invalid cluster id          | Old logs exist                     | Delete log.dirs                 |
